<a href="https://colab.research.google.com/github/LucasGABernardo/TrabalhoIA/blob/main/TrabalhoFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)
np.random.seed(42)

In [ ]:
df = pd.read_csv('retail_fraud_detection_100k.csv')

In [ ]:
print(f"\nDimensões do Dataset: {df.shape[0]} linhas por {df.shape[1]} colunas.")

print("\n--- Distribuição Absoluta e Percentual de Fraudes ---")
contagem_fraude = df['fraud_flag'].value_counts()
proporcao_fraude = df['fraud_flag'].value_counts(normalize=True) * 100
for classe, qtd in contagem_fraude.items():
    print(f"Classe {classe}: {qtd} registros ({proporcao_fraude[classe]:.2f}%)")

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='fraud_flag', palette='Reds_r')
plt.title('Distribuição da Variável Target (fraud_flag)', fontsize=12, fontweight='bold')
plt.xlabel('0 = Transação Legítima | 1 = Fraude')
plt.ylabel('Contagem')
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='fraud_flag', y='transaction_amount', palette='Set1')
plt.title('Análise de Outliers: Valor da Transação vs Indicador de Fraude', fontsize=12, fontweight='bold')
plt.yscale('log')
plt.show()

In [ ]:
colunas_numericas = [
    'transaction_amount', 'transaction_frequency_24h', 'avg_transaction_amount_7d',
    'failed_transaction_count_24h', 'account_age_days'
]

colunas_categoricas = [
    'payment_method', 'device_type', 'location', 'merchant_category'
]

colunas_flags = [
    'is_international', 'previous_fraud_flag', 'unusual_amount_flag',
    'unusual_location_flag', 'multiple_transactions_short_time',
    'high_risk_device_flag', 'velocity_flag'
]

transformer_numerico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

transformer_categorico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', transformer_numerico, colunas_numericas),
    ('cat', transformer_categorico, colunas_categoricas),
    ('flags', 'passthrough', colunas_flags)
])

X = df[colunas_numericas + colunas_categoricas + colunas_flags]
y = df['fraud_flag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Base de treinamento estruturada com {X_train.shape[0]} amostras.")
print(f"Base de testes estruturada com {X_test.shape[0]} amostras.")

In [ ]:
modelos_supervisionados = {
    'Regressão Logística': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42, n_jobs=-1),
    'KNN (K=5)': KNeighborsClassifier(n_neighbors=5)
}

resultados_finais = {}

for nome, modelo in modelos_supervisionados.items():
    pipeline_modelo = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', modelo)])

    print(f"Treinando modelo: {nome}...")
    pipeline_modelo.fit(X_train, y_train)

    predicoes = pipeline_modelo.predict(X_test)

    if hasattr(pipeline_modelo, "predict_proba"):
        probabilidades = pipeline_modelo.predict_proba(X_test)[:, 1]
    else:
        probabilidades = predicoes

    resultados_finais[nome] = {
        'Acurácia': accuracy_score(y_test, predicoes),
        'Precisão': precision_score(y_test, predicoes, zero_division=0),
        'Recall': recall_score(y_test, predicoes),
        'F1-Score': f1_score(y_test, predicoes),
        'ROC-AUC': roc_auc_score(y_test, probabilidades),
        'Matriz_Confusao': confusion_matrix(y_test, predicoes)
    }

print("\nTodos os modelos supervisionados foram avaliados!")

In [ ]:
X_train_clean = preprocessor.fit_transform(X_train)
X_test_clean = preprocessor.transform(X_test)

taxa_contaminacao = y_train.mean()

modelo_anomalia = IsolationForest(contamination=taxa_contaminacao, random_state=42, n_jobs=-1)
modelo_anomalia.fit(X_train_clean)

predicoes_if_cruas = modelo_anomalia.predict(X_test_clean)

predicoes_if = np.where(predicoes_if_cruas == -1, 1, 0)

resultados_finais['Isolation Forest (Não-Superv.)'] = {
    'Acurácia': accuracy_score(y_test, predicoes_if),
    'Precisão': precision_score(y_test, predicoes_if, zero_division=0),
    'Recall': recall_score(y_test, predicoes_if),
    'F1-Score': f1_score(y_test, predicoes_if),
    'ROC-AUC': roc_auc_score(y_test, predicoes_if),
    'Matriz_Confusao': confusion_matrix(y_test, predicoes_if)
}

print("Detecção de anomalias calculada e anexada com sucesso!")

In [ ]:
df_ranking = pd.DataFrame(resultados_finais).T.drop(columns=['Matriz_Confusao'])
print("======= QUADRO DEFINITIVO DE PERFORMANCE =======")
print(df_ranking.round(4))

fig, axes = plt.subplots(1, 4, figsize=(24, 5))
for i, (nome_modelo, payload) in enumerate(resultados_finais.items()):
    display_matriz = ConfusionMatrixDisplay(
        confusion_matrix=payload['Matriz_Confusao'],
        display_labels=['Legítima', 'Fraude']
    )
    display_matriz.plot(cmap='YlOrRd', values_format='d', ax=axes[i], colorbar=False)
    axes[i].set_title(nome_modelo, fontsize=11, fontweight='bold')
    axes[i].grid(False)

plt.suptitle('Análise Comparativa: Matrizes de Confusão (Dados de Teste)', fontsize=15, fontweight='bold', y=1.06)
plt.tight_layout()
plt.show()